# 모델 있고 없고 비교 — 실용성 검증

`train_emotion_classifier.ipynb`로 학습한 KoELECTRA 감정 분류기를 실제로 붙일 가치가 있는지 확인한다.

## 비교 대상

- **(A) 없는 경우** — Gemini(`gemini-2.5-flash`) 단독 제로샷 감정 분류.
  지금 `hyope-ai-server/llm_analyzer.py`의 프로덕션 프롬프트는 6종 감정 분류를 아예 하지 않고
  depressionScore 등 연속 점수만 낸다. 여기서는 "커스텀 모델 없이 LLM한테 감정 라벨까지 시키면
  어떨까"를 순수 제로샷 프롬프트로 재현한다.
- **(B) 있는 경우** — `model_output/`에 저장된 우리 KoELECTRA 모델(`predict.py`의 `EmotionPredictor`).

둘 다 같은 데이터로 채점해야 공정하므로 `train/evaluate.py`의 `split_aihub_holdout()`이 만드는
**보류 테스트셋**(학습에 전혀 쓰이지 않은 AI Hub 데이터, 1,015건)에서 뽑는다.

## 측정 항목

1. 정확도 / 매크로 F1 (레이블별)
2. 두 방식의 예측 일치율, 서로 다르게 틀리는 사례
3. 건당 응답 시간 (실용성 — 배포 시 지연에 영향)

## 왜 전체 1,015건이 아니라 샘플인가

Gemini 호출은 시간과 (약간의) 비용이 든다. 기본은 계층화 샘플 `SAMPLE_SIZE`건만 쓰고,
필요하면 아래 셀에서 값을 키워서 전체 보류셋으로 확장할 수 있다.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI

# train/ 모듈 임포트 경로 (노트북은 notebooks/ 에서 실행된다고 가정)
TRAIN_DIR = Path.cwd().parent / "train"
sys.path.insert(0, str(TRAIN_DIR))

from config import ID2LABEL, LABEL2ID, LABELS, TrainConfig
from evaluate import split_aihub_holdout
from predict import EmotionPredictor

# GEMINI_API_KEY는 hyope-ai-server/.env에 있다 (별도 repo라 경로를 명시)
load_dotenv(Path.cwd().parent.parent / "hyope-ai-server" / ".env")

config = TrainConfig()
MODEL_PATH = config.output_dir  # 학습 노트북과 동일한 model_output/
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def get_gemini_client() -> OpenAI:
    return OpenAI(
        api_key=os.getenv("GEMINI_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )


print(f"모델 경로: {MODEL_PATH}")
print(f"device: {DEVICE}")
print(f"GEMINI_API_KEY 설정됨: {bool(os.getenv('GEMINI_API_KEY'))}")

## 1. 보류 테스트셋에서 계층화 샘플 뽑기

`split_aihub_holdout()`은 학습 노트북·`evaluate.py`와 **같은 seed**를 쓰므로,
여기서 뽑는 보류셋은 학습에 전혀 섞이지 않은 데이터임이 보장된다.

In [ ]:
from sklearn.model_selection import train_test_split

SAMPLE_SIZE = 200  # Gemini 호출 비용/시간 감안한 기본값. 늘리면 holdout 전체(1,015)까지 가능

_, holdout = split_aihub_holdout(config)
print(f"보류 테스트셋 전체: {len(holdout)}건")

if SAMPLE_SIZE >= len(holdout):
    sample = holdout
else:
    sample, _ = train_test_split(
        holdout,
        train_size=SAMPLE_SIZE,
        random_state=config.seed,
        stratify=[s["label"] for s in holdout],
    )

texts = [s["text"] for s in sample]
true_labels = [s["label"] for s in sample]
print(f"평가 샘플: {len(sample)}건")
pd.Series([ID2LABEL[l] for l in true_labels]).value_counts().reindex(LABELS)

## 2. (A) 없는 경우 — Gemini 단독 제로샷 분류

프로덕션 프롬프트와 별개로, "감정 라벨만" 물어보는 최소 프롬프트를 사용한다.
`gemini-2.5-flash`는 내부적으로 thinking 토큰을 꽤 소모하므로 `max_tokens`를 넉넉히 줘야 한다
(100으로 테스트했다가 JSON이 중간에 잘려서 파싱 실패하는 걸 확인함 → 500으로 설정).

In [ ]:
import json

ZS_PROMPT = f"""다음 문장의 감정을 아래 6종 중 하나로만 분류하세요: {", ".join(LABELS)}

설명, markdown, 코드블록 없이 아래 JSON 형식만 반환하세요.
{{"emotion": "<6종 중 하나>"}}"""


def classify_with_gemini(client: OpenAI, text: str, max_retries: int = 2) -> dict:
    """단일 텍스트를 Gemini 제로샷으로 분류. 실패 시 emotion=None."""
    last_raw = ""
    latency = 0.0
    for attempt in range(max_retries + 1):
        start = time.perf_counter()
        try:
            resp = client.chat.completions.create(
                model="gemini-2.5-flash",
                messages=[
                    {"role": "system", "content": ZS_PROMPT},
                    {"role": "user", "content": text},
                ],
                temperature=0,
                max_tokens=500,
            )
        except Exception as e:
            latency = time.perf_counter() - start
            if attempt < max_retries:
                continue
            return {"emotion": None, "latency": latency, "error": str(e)}
        latency = time.perf_counter() - start

        last_raw = resp.choices[0].message.content or ""
        cleaned = last_raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[-1]
            cleaned = cleaned.rsplit("```", 1)[0].strip()

        try:
            parsed = json.loads(cleaned)
            emotion = parsed.get("emotion")
            if emotion in LABEL2ID:
                return {"emotion": emotion, "latency": latency, "error": None}
        except json.JSONDecodeError:
            pass

        if attempt >= max_retries:
            return {"emotion": None, "latency": latency, "error": f"파싱 실패: {last_raw[:80]}"}
    return {"emotion": None, "latency": latency, "error": "unreachable"}

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm import tqdm

client = get_gemini_client()

gemini_results = [None] * len(texts)
with ThreadPoolExecutor(max_workers=6) as pool:
    futures = {pool.submit(classify_with_gemini, client, t): i for i, t in enumerate(texts)}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Gemini 제로샷"):
        i = futures[future]
        gemini_results[i] = future.result()

gemini_failures = [r for r in gemini_results if r["emotion"] is None]
print(f"실패: {len(gemini_failures)}건 / {len(gemini_results)}건")
if gemini_failures:
    print("실패 예시:", gemini_failures[0]["error"])

## 3. (B) 있는 경우 — 우리 모델(KoELECTRA)

같은 `texts`로 `EmotionPredictor`를 돌린다. 로컬 배치 추론이라 Gemini보다 훨씬 빠를 것이다.

In [ ]:
predictor = EmotionPredictor(MODEL_PATH, device=DEVICE)

start = time.perf_counter()
ours_raw = predictor.predict_batch(texts)
ours_total_time = time.perf_counter() - start
ours_results = [{"emotion": r["emotion"], "latency": ours_total_time / len(texts)} for r in ours_raw]

print(f"우리 모델 총 소요: {ours_total_time:.2f}s ({len(texts)}건, 평균 {ours_total_time / len(texts) * 1000:.1f}ms/건)")

## 4. 채점 — 정확도 / 매크로 F1

Gemini가 실패(파싱 실패 등)한 샘플은 오답 처리한다 — 실제 서비스에서 실패하면
그 값을 못 쓰는 것과 결과적으로 같기 때문.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

DUMMY = -1  # 실패(None)는 어떤 정답과도 안 맞는 더미 id로 처리 → 자동 오답


def to_pred_ids(results: list[dict]) -> list[int]:
    return [LABEL2ID.get(r["emotion"], DUMMY) for r in results]


gemini_pred = to_pred_ids(gemini_results)
ours_pred = to_pred_ids(ours_results)

label_ids = list(range(len(LABELS)))

print("=== (A) Gemini 제로샷 ===")
print(classification_report(true_labels, gemini_pred, labels=label_ids, target_names=LABELS, digits=3, zero_division=0))

print("=== (B) 우리 모델 ===")
print(classification_report(true_labels, ours_pred, labels=label_ids, target_names=LABELS, digits=3, zero_division=0))

summary = pd.DataFrame({
    "Gemini 제로샷": {
        "정확도": accuracy_score(true_labels, gemini_pred),
        "매크로 F1": f1_score(true_labels, gemini_pred, average="macro", zero_division=0),
        "평균 응답시간(ms)": np.mean([r["latency"] for r in gemini_results]) * 1000,
    },
    "우리 모델": {
        "정확도": accuracy_score(true_labels, ours_pred),
        "매크로 F1": f1_score(true_labels, ours_pred, average="macro", zero_division=0),
        "평균 응답시간(ms)": np.mean([r["latency"] for r in ours_results]) * 1000,
    },
}).T
summary

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

_available = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _font in ["Malgun Gothic", "NanumGothic", "NanumBarunGothic", "AppleGothic"]:
    if _font in _available:
        plt.rcParams["font.family"] = _font
        break
else:
    print("경고: 한글 폰트를 찾지 못했습니다. 차트의 한글이 깨질 수 있습니다.")
plt.rcParams["axes.unicode_minus"] = False

# 강조(우리 모델) / 대조(Gemini 베이스라인) — 우리 모델이 비교의 주인공이라 강조색을 주고
# Gemini는 중립색으로 맥락만 제공 (학습 노트북의 팔레트 규칙과 동일한 톤 유지)
OURS, BASELINE = "#2a78d6", "#b5b3ab"
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"


def style_axes(ax):
    ax.set_facecolor(SURFACE)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, labelcolor=INK_2)

In [ ]:
report_a = classification_report(true_labels, gemini_pred, labels=label_ids, target_names=LABELS, output_dict=True, zero_division=0)
report_b = classification_report(true_labels, ours_pred, labels=label_ids, target_names=LABELS, output_dict=True, zero_division=0)

per_label_f1 = pd.DataFrame({
    "Gemini 제로샷": [report_a[label]["f1-score"] for label in LABELS],
    "우리 모델": [report_b[label]["f1-score"] for label in LABELS],
}, index=LABELS)

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(LABELS))
height = 0.36
ax.barh(y + height / 2, per_label_f1["Gemini 제로샷"], height=height, color=BASELINE,
        label="Gemini 제로샷", edgecolor="white", linewidth=1)
ax.barh(y - height / 2, per_label_f1["우리 모델"], height=height, color=OURS,
        label="우리 모델", edgecolor="white", linewidth=1)
ax.set_yticks(y, LABELS)
ax.set_xlim(0, 1.05)
style_axes(ax)
ax.set_title("레이블별 F1 — Gemini 제로샷 vs 우리 모델", color=INK, loc="left", fontsize=12)
ax.legend(frameon=False, loc="lower right", labelcolor=INK_2)
for yi, (a, b) in enumerate(zip(per_label_f1["Gemini 제로샷"], per_label_f1["우리 모델"])):
    ax.text(a + 0.01, yi + height / 2, f"{a:.2f}", va="center", fontsize=8.5, color=INK_2)
    ax.text(b + 0.01, yi - height / 2, f"{b:.2f}", va="center", fontsize=8.5, color=INK_2)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

metrics = ["정확도", "매크로 F1"]
x = np.arange(len(metrics))
width = 0.32
axes[0].bar(x - width / 2, summary.loc["Gemini 제로샷", metrics], width=width, color=BASELINE,
            label="Gemini 제로샷", edgecolor="white")
axes[0].bar(x + width / 2, summary.loc["우리 모델", metrics], width=width, color=OURS,
            label="우리 모델", edgecolor="white")
axes[0].set_xticks(x, metrics)
axes[0].set_ylim(0, 1.05)
style_axes(axes[0])
axes[0].set_title("정확도 / 매크로 F1", color=INK, loc="left", fontsize=11)
axes[0].legend(frameon=False, labelcolor=INK_2)

axes[1].bar(["Gemini 제로샷", "우리 모델"], summary["평균 응답시간(ms)"], color=[BASELINE, OURS], edgecolor="white")
style_axes(axes[1])
axes[1].set_title("평균 응답시간 (ms/건)", color=INK, loc="left", fontsize=11)
for i, v in enumerate(summary["평균 응답시간(ms)"]):
    axes[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9, color=INK_2)

plt.tight_layout()
plt.show()

## 5. 두 방식이 갈리는 지점

정답 기준 채점 말고, **서로 다르게 예측한 사례**를 보면 실제로 뭐가 다른지 감이 온다.

In [ ]:
diff_df = pd.DataFrame({
    "텍스트": texts,
    "정답": [ID2LABEL[t] for t in true_labels],
    "Gemini": [r["emotion"] or "(실패)" for r in gemini_results],
    "우리모델": [r["emotion"] for r in ours_results],
})

agree_rate = (diff_df["Gemini"] == diff_df["우리모델"]).mean()
print(f"두 방식 예측 일치율: {agree_rate:.1%}")

only_ours_correct = diff_df[(diff_df["우리모델"] == diff_df["정답"]) & (diff_df["Gemini"] != diff_df["정답"])]
only_gemini_correct = diff_df[(diff_df["Gemini"] == diff_df["정답"]) & (diff_df["우리모델"] != diff_df["정답"])]

print(f"우리 모델만 맞춤: {len(only_ours_correct)}건 / Gemini만 맞춤: {len(only_gemini_correct)}건")
with pd.option_context("display.max_colwidth", 60):
    display(only_ours_correct.head(8))
    display(only_gemini_correct.head(8))

## 6. 실용성 판단 가이드

숫자를 채운 뒤 아래 기준으로 판단한다 (임의 기준이니 팀 기준에 맞게 조정):

- **매크로 F1 차이가 크지 않다면(예: 0.03 미만)** → Gemini가 어차피 통화마다 호출되므로,
  같은 호출에 감정 분류 필드를 하나 추가하는 게 더 간단할 수 있다. 별도 모델 배포/유지보수 비용을 아낀다.
- **우리 모델이 뚜렷하게 더 정확하다면** → 노인 발화 도메인(AI Hub 데이터)에 특화 학습한 효과가
  실제로 있다는 뜻. 응답시간 차이(로컬 추론이 훨씬 빠름)까지 고려하면 배포 가치가 더 커진다.
- **Gemini 실패율(파싱 실패 등)이 유의미하다면** → 프로덕션에서도 같은 실패가 날 수 있다는 뜻이니
  이것만으로도 로컬 모델을 쓸 이유가 된다 (실패 시 폴백이 없으면 그 필드가 통째로 빠진다).
- **`SAMPLE_SIZE`를 키워서(예: 전체 1,015건) 다시 돌리면** 신뢰구간이 좁아져 판단이 더 명확해진다.

이 노트북은 판단에 필요한 숫자를 만드는 것까지가 목적이고, 최종 결정(배포 여부)은 팀 회의에서.